In [2]:
%pip install xgboost


   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
   ------------- -------------------------- 16.0/48.9 MB 82.8 MB/s eta 0:00:01
   ------------------------------ --------- 37.5/48.9 MB 94.2 MB/s eta 0:00:01
   ---------------------------------------  48.8/48.9 MB 95.9 MB/s eta 0:00:01
   ---------------------------------------- 48.9/48.9 MB 78.9 MB/s eta 0:00:00


In [3]:
import pandas as pd
import numpy as np

from xgboost import XGBRegressor
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


# ============================================================
# 1. Load dataset
# ============================================================

DATA_PATH = r"C:\Users\muskan\OneDrive\Desktop\effisense\data\processed\mouse_productivity_scores_reduced.csv"

df = pd.read_csv(DATA_PATH)


# ============================================================
# 2. Features and target
# ============================================================

REDUCED_FEATURES = [
    "idle_percentage",
    "pause_count",
    "mean_pause",
    "mean_velocity",
    "total_distance",
    "mean_distance",
    "velocity_cv",
    "velocity_entropy",
    "smoothness",
    "mean_acceleration",
    "std_acceleration",
    "mean_direction_change",
    "direction_entropy",
    "num_events"
]

TARGET = "productivity_score"


X = df[REDUCED_FEATURES].copy()
y = df[TARGET].copy()

# Handle missing values
X = X.fillna(X.median())


# ============================================================
# 3. Create unique user-session groups
# ============================================================

groups = (
    df["user"].astype(str)
    + "_"
    + df["session"].astype(str)
)


# ============================================================
# 4. 5-Fold GroupKFold
# ============================================================

gkf = GroupKFold(n_splits=5)

results = []


# ============================================================
# 5. Train XGBoost on each fold
# ============================================================

for fold, (train_idx, test_idx) in enumerate(
    gkf.split(X, y, groups=groups),
    start=1
):

    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]


    # --------------------------------------------------------
    # XGBoost model
    # --------------------------------------------------------

    xgb = XGBRegressor(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )


    # --------------------------------------------------------
    # Train
    # --------------------------------------------------------

    xgb.fit(X_train, y_train)


    # --------------------------------------------------------
    # Predict
    # --------------------------------------------------------

    y_pred_raw = xgb.predict(X_test)

    # Productivity score must be between 0 and 100
    y_pred = np.clip(y_pred_raw, 0, 100)


    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------

    mae = mean_absolute_error(y_test, y_pred)

    mse = mean_squared_error(y_test, y_pred)

    rmse = np.sqrt(mse)

    r2 = r2_score(y_test, y_pred)


    results.append({
        "fold": fold,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2
    })


# ============================================================
# 6. Results
# ============================================================

results_df = pd.DataFrame(results)

print("\nXGBoost — 5-Fold Session-Level Cross-Validation")
print("=" * 60)

print(results_df.to_string(index=False))


# ============================================================
# 7. Mean ± Standard Deviation
# ============================================================

print("\nMean ± Standard Deviation")
print("=" * 60)

for metric in ["MAE", "MSE", "RMSE", "R2"]:

    mean_value = results_df[metric].mean()
    std_value = results_df[metric].std()

    print(
        f"{metric:5s}: "
        f"{mean_value:.4f} ± {std_value:.4f}"
    )


XGBoost — 5-Fold Session-Level Cross-Validation
 fold      MAE      MSE     RMSE       R2
    1 0.798205 1.083827 1.041070 0.998234
    2 0.919045 1.947815 1.395642 0.996769
    3 0.774403 1.036057 1.017869 0.997104
    4 0.770478 0.982788 0.991356 0.998332
    5 0.770084 0.999205 0.999602 0.998239

Mean ± Standard Deviation
MAE  : 0.8064 ± 0.0640
MSE  : 1.2099 ± 0.4143
RMSE : 1.0891 ± 0.1724
R2   : 0.9977 ± 0.0007
